In [3]:
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow.keras import layers, models

# 1. Load the dataset
# We split it: 80% for training, 20% for validation
(train_ds, val_ds), ds_info = tfds.load(
    'cats_vs_dogs',
    split=['train[:80%]', 'train[80%:]'],
    as_supervised=True,  # Returns (image, label) tuples
    with_info=True
)

# 2. Preprocessing Function
def preprocess_image(image, label):
    image = tf.image.resize(image, (150, 150)) # Resize to standard size
    image = tf.cast(image, tf.float32) / 255.0  # Normalize to [0, 1]
    return image, label

# 3. Build the Data Pipeline
# Shuffle, Batch, and Autotune for performance
BATCH_SIZE = 32
train_ds = train_ds.map(preprocess_image).shuffle(1000).batch(BATCH_SIZE).prefetch(buffer_size=tf.data.AUTOTUNE)
val_ds = val_ds.map(preprocess_image).batch(BATCH_SIZE).prefetch(buffer_size=tf.data.AUTOTUNE)

# 4. Build the CNN Model
model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(150, 150, 3)),
    layers.MaxPooling2D(2, 2),
    
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D(2, 2),
    
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D(2, 2),
    
    layers.Flatten(),
    layers.Dense(512, activation='relu'),
    layers.Dense(1, activation='sigmoid') # Binary classification
])

# 5. Compile and Train
model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

print("Training starting...")
model.fit(train_ds, epochs=10, validation_data=val_ds)

# 6. Save the model
model.save('cats_vs_dogs_tfds.h5')
print("Model saved successfully.")

Dl Completed...: 0 url [00:00, ? url/s]
Generating splits...:   0%|                                                                 | 0/1 [00:00<?, ? splits/s]
Generating train examples...: 0 examples [00:00, ? examples/s]


KeyError: "There is no item named 'PetImages\\\\Cat\\\\0.jpg' in the archive"

In [ ]:
pip install importlib_resources

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np

# 1. Load the built-in CIFAR-10 dataset
print("Loading CIFAR-10 dataset...")
(x_train_all, y_train_all), (x_test_all, y_test_all) = tf.keras.datasets.cifar10.load_data()

# 2. Filter for Cats (label 3) and Dogs (label 5)
# This creates a boolean mask to pick only the images we want
train_mask = np.isin(y_train_all, [3, 5]).flatten()
test_mask = np.isin(y_test_all, [3, 5]).flatten()

x_train, y_train = x_train_all[train_mask], y_train_all[train_mask]
x_test, y_test = x_test_all[test_mask], y_test_all[test_mask]

# 3. Preprocess the data
# Map Cat(3) to 0 and Dog(5) to 1
y_train = np.where(y_train == 3, 0, 1)
y_test = np.where(y_test == 3, 0, 1)

# Normalize pixel values to [0, 1]
x_train, x_test = x_train / 255.0, x_test / 255.0

print(f"Training set size: {len(x_train)} images")
print(f"Test set size: {len(x_test)} images")

# 4. Build the CNN Model
model = models.Sequential([
    # CIFAR-10 images are 32x32 pixels with 3 color channels (RGB)
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(32, 32, 3)),
    layers.MaxPooling2D((2, 2)),
    
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(1, activation='sigmoid') # Sigmoid for binary output
])

# 5. Compile and Train
model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

print("\nStarting Training...")
model.fit(x_train, y_train, epochs=10, validation_data=(x_test, y_test))

# 6. Evaluate
loss, acc = model.evaluate(x_test, y_test)
print(f"\nFinal Accuracy: {acc*100:.2f}%")

In [ ]:
import numpy as np
from tensorflow.keras.preprocessing import image

def predict_animal(img_path):
    # 1. Load the image and resize it to 32x32 (CIFAR-10 size)
    img = image.load_img(img_path, target_size=(32, 32))
    
    # 2. Convert image to array and normalize
    img_array = image.img_to_array(img)
    img_array = img_array / 255.0  # Same normalization as training
    
    # 3. Add a fourth dimension (batch size) -> (1, 32, 32, 3)
    img_array = np.expand_dims(img_array, axis=0)
    
    # 4. Make the prediction
    prediction = model.predict(img_array)
    
    # 5. Interpret the result
    # Remember: Cat was 0, Dog was 1
    if prediction[0][0] > 0.5:
        print(f"Confidence: {prediction[0][0]*100:.2f}% -> It's a DOG! 🐶")
    else:
        print(f"Confidence: {(1-prediction[0][0])*100:.2f}% -> It's a CAT! 🐱")

# Usage: 
# Replace 'my_pet.jpg' with the actual path to the image you downloaded
# predict_animal('my_pet.jpg')

In [ ]:
predict_animal(r"C:\Users\Laksh\Downloads\images.jpg")